# CropSure Research Notebook 02: Van der Plank Epidemic Differential Simulation

## Theoretical Foundation: Van der Plank Epidemiology
In 1963, J.E. Van der Plank published his landmark treatise *Plant Diseases: Epidemics and Control*, formulating the mathematical foundations of botanical epidemiology. Plant disease epidemics operate analogously to compound financial interest, where initial inoculum produces infectious tissue that recursively generates secondary inoculum.

Unlike classic non-delayed logistic formulations, polycyclic plant pathogens (such as *Phytophthora infestans* Late Blight and *Alternaria solani* Early Blight) exhibit a distinct biological latency period ($p$). An infected tissue region requires $p$ days before becoming sporulating and infectious. Van der Plank expressed this delayed differential dynamic as:

$$\frac{dy(t)}{dt} = R_c \cdot y(t - p) \cdot [1 - y(t)]$$

where:
- $y(t)$ is the proportion of infected host tissue (severity index $\in [0, 1]$).
- $1 - y(t)$ is the remaining susceptible, healthy foliar tissue.
- $y(t - p)$ is the infectious tissue that initiated sporulation $p$ days prior.
- $R_c$ is the corrected basic infection rate under prevailing meteorological conditions.

## Research Objectives
1. Formulate a numerically stabilized delay-differential system incorporating environmental modifiers for temperature ($T$), relative humidity ($RH$), and precipitation ($P$).
2. Execute Latin Hypercube Sampling (LHS) across 5,000 parameter spaces to construct an empirical synthetic training corpus for machine learning forecasters.
3. Perform comparative scenario modeling: Favorable, Unfavorable, Treated (fungicidal remediation), and Untreated epidemics.
4. Quantify Area Under the Disease Progress Curve (AUDPC) and infection inflection thresholds.

In [ ]:
# Cell 02: Scientific Imports
import os
import sys
import time
import math
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import solve_ivp
from scipy.stats import qmc

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print(f"[IMPORT] NumPy: {np.__version__} | SciPy: {sys.modules['scipy'].__version__} | Pandas: {pd.__version__}")

In [ ]:
# Cell 03: Seed Fixation & Parameter Dataclass Definition
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Establish Directory Context
NOTEBOOK_DIR = Path(os.path.abspath("")).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class ModelParameters:
    """Parameters governing the Van der Plank epidemiological ODE system."""
    r0: float = 0.20           # Intrinsic apparent infection rate (day^-1)
    latent_period: float = 4.0 # Incubation latency (days)
    removal_rate: float = 0.01 # Natural tissue senescence/removal rate (day^-1)
    y0: float = 0.01           # Initial primary inoculum fraction
    t_opt: float = 22.0        # Optimal temperature (°C)
    t_sigma: float = 8.0       # Thermal Gaussian curve width
    rh_threshold: float = 60.0 # Relative humidity activation threshold (%)
    rh_range: float = 40.0     # Relative humidity scaling denominator
    sim_days: int = 60         # Simulation duration (days)

params_default = ModelParameters()
print("[CONFIG] Default Epidemiological Parameters:")
for k, v in params_default.__dict__.items():
    print(f"  - {k:16s}: {v}")

## Mathematical Formulation & Environmental Modifiers

Under field microclimate conditions, the actual pathogen reproductive rate is modulated by temperature ($T$), relative humidity ($RH$), and leaf wetness/precipitation ($P$):

### 1. Thermal Modifier $f(T)$
Pathogen fungal sporulation follows a Gaussian response envelope centered around optimal temperature $T_{\text{opt}} = 22^\circ\text{C}$:
$$f(T) = \exp\left(-\left[\frac{T - 22.0}{8.0}\right]^2\right)$$

### 2. Relative Humidity Modifier $f(RH)$
Foliar fungal conidia germination requires high atmospheric moisture ($RH > 60\%$):
$$f(RH) = \max\left(0.0, \frac{RH - 60.0}{40.0}\right)$$

### 3. Precipitation & Free Moisture Factor $f(P)$
Rainfall induces splash dispersal of spores across neighboring foliage:
$$f(P) = 1.0 + 0.50 \cdot \frac{P}{P + 10.0}$$

### 4. Complete Governed System
The compound differential equation becomes:
$$\frac{dy(t)}{dt} = r_0 \cdot f(T) \cdot f(RH) \cdot f(P) \cdot y(t - p) \cdot [1 - y(t)] - \mu \cdot y(t)$$

where $\mu$ is the natural tissue necrotic removal/senescence rate.

In [ ]:
# Cell 05: VanDerPlankODE Implementation with Numerical Stability Safeguards
class VanDerPlankODE:
    """
    Van der Plank delay-differential epidemiological simulator.
    Utilizes continuous history interpolation and SciPy solve_ivp.
    """
    def __init__(self, params: ModelParameters = None):
        self.params = params if params is not None else ModelParameters()
        
    def temperature_modifier(self, T: float) -> float:
        """Gaussian thermal envelope modifier."""
        return math.exp(-((T - self.params.t_opt) / self.params.t_sigma) ** 2)
    
    def humidity_modifier(self, RH: float) -> float:
        """Linear threshold relative humidity modifier."""
        return max(0.0, min(1.0, (RH - self.params.rh_threshold) / self.params.rh_range))
    
    def precipitation_modifier(self, P: float) -> float:
        """Splash dispersal moisture modifier."""
        return 1.0 + 0.50 * (P / (P + 10.0))
    
    def effective_rate(self, r0: float, T: float, RH: float, P: float, treated: bool = False, treatment_day: float = 14.0, t_current: float = 0.0) -> float:
        """Computes environmentally coupled effective infection rate."""
        f_T = self.temperature_modifier(T)
        f_RH = self.humidity_modifier(RH)
        f_P = self.precipitation_modifier(P)
        r_eff = r0 * f_T * f_RH * f_P
        
        # Fungicidal intervention reduces infection rate by 60% (factor 0.40)
        if treated and t_current >= treatment_day:
            r_eff *= 0.40
            
        return r_eff
    
    def simulate(
        self,
        days: int = 60,
        T: float = 24.0,
        RH: float = 85.0,
        P: float = 10.0,
        r0: float = 0.20,
        latent_period: float = 4.0,
        treated: bool = False,
        treatment_day: float = 14.0,
        dt: float = 0.25
    ) -> pd.DataFrame:
        """
        Solves the delayed epidemic system with step-by-step history interpolation.
        """
        timesteps = int(days / dt) + 1
        t_eval = np.linspace(0, days, timesteps)
        
        # State trajectory storage
        y_history = np.zeros(timesteps)
        y_history[0] = self.params.y0
        
        def get_delayed_y(current_t, delay):
            delayed_t = current_t - delay
            if delayed_t <= 0.0:
                return self.params.y0
            idx = int(delayed_t / dt)
            idx = min(idx, timesteps - 1)
            return y_history[idx]
        
        # Numerical integration using Runge-Kutta 4th-Order (RK4) with delay buffer
        for i in range(timesteps - 1):
            t_curr = t_eval[i]
            y_curr = y_history[i]
            
            # Sub-step RK4 evaluation
            def ode_rhs(t, y):
                y_delayed = get_delayed_y(t, latent_period)
                r_eff = self.effective_rate(r0, T, RH, P, treated, treatment_day, t)
                # Stiff numerical safeguard: clamp y to [0.0, 1.0]
                y_clamped = max(0.0, min(1.0, y))
                y_del_clamped = max(0.0, min(1.0, y_delayed))
                # dy/dt = r_eff * y(t - p) * (1 - y(t)) - mu * y(t)
                growth = r_eff * y_del_clamped * (1.0 - y_clamped)
                removal = self.params.removal_rate * y_clamped
                return growth - removal
            
            k1 = ode_rhs(t_curr, y_curr)
            k2 = ode_rhs(t_curr + dt * 0.5, y_curr + 0.5 * dt * k1)
            k3 = ode_rhs(t_curr + dt * 0.5, y_curr + 0.5 * dt * k2)
            k4 = ode_rhs(t_curr + dt, y_curr + dt * k3)
            
            y_next = y_curr + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
            # Clamp bounds to [0.0, 1.0] to guarantee biological plausibility
            y_history[i + 1] = max(0.0, min(1.0, y_next))
            
        # Sub-sample to integer days
        day_indices = [int(d / dt) for d in range(days + 1)]
        sampled_days = np.array(range(days + 1))
        sampled_severities = y_history[day_indices]
        
        # Compute AUDPC (Trapezoidal Rule)
        audpc_cum = np.zeros(days + 1)
        for d in range(1, days + 1):
            audpc_cum[d] = audpc_cum[d-1] + 0.5 * (sampled_severities[d] + sampled_severities[d-1])
            
        # Daily growth rates
        daily_rates = np.diff(sampled_severities, prepend=0.0)
        
        df = pd.DataFrame({
            "day": sampled_days,
            "severity": np.round(sampled_severities, 5),
            "growth_rate": np.round(daily_rates, 5),
            "audpc": np.round(audpc_cum, 3),
            "temperature": T,
            "humidity": RH,
            "precipitation": P,
            "infection_rate": r0,
            "latent_period": latent_period,
            "treated": 1 if treated else 0
        })
        return df

simulator = VanDerPlankODE()
print("[ODE] VanDerPlankODE numerical engine successfully initialized.")

In [ ]:
# Cell 06: Generate Single Trajectory & Epidemiological Profile
baseline_df = simulator.simulate(
    days=60,
    T=24.0,
    RH=85.0,
    P=15.0,
    r0=0.22,
    latent_period=4.0,
    treated=False
)

inflection_idx = baseline_df["growth_rate"].idxmax()
inflection_day = baseline_df.loc[inflection_idx, "day"]
peak_rate = baseline_df.loc[inflection_idx, "growth_rate"]
final_audpc = baseline_df["audpc"].iloc[-1]

print("==================================================")
print("      Baseline Epidemic Trajectory Summary        ")
print("==================================================")
print(f"  - Peak Infection Growth Day  : Day {inflection_day:.0f}")
print(f"  - Maximum Daily Growth Rate  : {peak_rate:.4f} day^-1")
print(f"  - Terminal Severity (Day 60) : {baseline_df['severity'].iloc[-1]:.4f}")
print(f"  - Cumulative AUDPC           : {final_audpc:.2f} %-days")
print("==================================================")

fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:red'
ax1.set_xlabel('Epidemic Progress (Days)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Foliar Severity $y(t)$', color=color, fontsize=12, fontweight='bold')
line1 = ax1.plot(baseline_df['day'], baseline_df['severity'], color=color, lw=3, label='Disease Severity $y(t)$')
ax1.tick_params(axis='y', labelcolor=color)
ax1.axvline(inflection_day, color='gray', linestyle=':', lw=2, label=f'Inflection Point (Day {inflection_day})')
ax1.set_ylim(-0.02, 1.05)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Daily Growth Rate $dy/dt$', color=color, fontsize=12, fontweight='bold')
line2 = ax2.plot(baseline_df['day'], baseline_df['growth_rate'], color=color, lw=2, linestyle='--', label='Growth Rate $dy/dt$')
ax2.tick_params(axis='y', labelcolor=color)

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', frameon=True)

plt.title('Baseline Van der Plank Delay-Differential Epidemic Progression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 07: Generate 5,000 Synthetic Trajectories via Latin Hypercube Sampling (LHS)
NUM_TRAJECTORIES = 5000
SIM_DAYS = 60
print(f"[LHS] Initializing Latin Hypercube Sampling for {NUM_TRAJECTORIES} trajectories...")

# Sample 5 dimensions: T, RH, P, r0, latent_period
sampler = qmc.LatinHypercube(d=5, seed=RANDOM_SEED)
lhs_samples = sampler.random(n=NUM_TRAJECTORIES)

# Map bounds to agronomic domains
# T: 15 to 35 °C
# RH: 40 to 100 %
# P: 0 to 50 mm
# r0: 0.05 to 0.40 day^-1
# latent_period: 2.0 to 7.0 days
l_bounds = [15.0, 40.0, 0.0, 0.05, 2.0]
u_bounds = [35.0, 100.0, 50.0, 0.40, 7.0]
scaled_samples = qmc.scale(lhs_samples, l_bounds, u_bounds)

start_sim_time = time.time()
trajectory_records = []

# 20% of synthetic runs simulate treated crops (fungicidal remediation at day 14)
treated_flags = np.random.RandomState(RANDOM_SEED).binomial(1, 0.20, size=NUM_TRAJECTORIES)

for traj_id in range(NUM_TRAJECTORIES):
    T_val = float(scaled_samples[traj_id, 0])
    RH_val = float(scaled_samples[traj_id, 1])
    P_val = float(scaled_samples[traj_id, 2])
    r0_val = float(scaled_samples[traj_id, 3])
    lp_val = float(scaled_samples[traj_id, 4])
    is_treated = bool(treated_flags[traj_id])
    
    traj_df = simulator.simulate(
        days=SIM_DAYS,
        T=T_val,
        RH=RH_val,
        P=P_val,
        r0=r0_val,
        latent_period=lp_val,
        treated=is_treated,
        treatment_day=14.0
    )
    traj_df["trajectory_id"] = traj_id
    trajectory_records.append(traj_df)

synthetic_df = pd.concat(trajectory_records, ignore_index=True)
elapsed_sim = time.time() - start_sim_time

# Reorder columns matching target schema
target_cols = [
    "trajectory_id",
    "day",
    "severity",
    "temperature",
    "humidity",
    "precipitation",
    "infection_rate",
    "latent_period",
    "treated"
]
synthetic_df = synthetic_df[target_cols]

print(f"[LHS] Successfully generated {len(synthetic_df):,} daily records across {NUM_TRAJECTORIES} trajectories in {elapsed_sim:.2f}s.")
print(synthetic_df.head())

In [ ]:
# Cell 08: Comparative Agronomic Scenario Simulation
scenarios = {
    "Favorable (Warm+Humid)": {
        "T": 28.0, "RH": 90.0, "P": 15.0, "r0": 0.28, "latent_period": 3.5, "treated": False
    },
    "Unfavorable (Cool+Dry)": {
        "T": 15.0, "RH": 45.0, "P": 0.0, "r0": 0.15, "latent_period": 6.0, "treated": False
    },
    "Treated (Remediation @ Day 14)": {
        "T": 26.0, "RH": 85.0, "P": 10.0, "r0": 0.25, "latent_period": 4.0, "treated": True
    },
    "Untreated Baseline": {
        "T": 26.0, "RH": 85.0, "P": 10.0, "r0": 0.25, "latent_period": 4.0, "treated": False
    }
}

scenario_results = {}
for name, s_params in scenarios.items():
    df_scen = simulator.simulate(
        days=60,
        T=s_params["T"],
        RH=s_params["RH"],
        P=s_params["P"],
        r0=s_params["r0"],
        latent_period=s_params["latent_period"],
        treated=s_params["treated"],
        treatment_day=14.0
    )
    scenario_results[name] = df_scen

# Multi-Panel Scenario Comparison Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette = {
    "Favorable (Warm+Humid)": "#d62728",
    "Untreated Baseline": "#ff7f0e",
    "Treated (Remediation @ Day 14)": "#2ca02c",
    "Unfavorable (Cool+Dry)": "#1f77b4"
}

# Panel A: Severity Trajectories
for name, df_scen in scenario_results.items():
    axes[0].plot(df_scen["day"], df_scen["severity"], label=name, color=palette[name], lw=2.5)

axes[0].axvline(14, color="green", linestyle=":", lw=1.5, label="Remediation Window (Day 14)")
axes[0].set_title("Epidemic Severity Trajectories across Agronomic Scenarios", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Simulation Day", fontsize=11)
axes[0].set_ylabel("Disease Severity Index [0, 1]", fontsize=11)
axes[0].set_ylim(-0.02, 1.02)
axes[0].legend(loc="upper left", frameon=True)
axes[0].grid(True, alpha=0.3)

# Panel B: Final Cumulative AUDPC Comparison
audpc_vals = [df_scen["audpc"].iloc[-1] for df_scen in scenario_results.values()]
scenario_names = list(scenario_results.keys())
bar_colors = [palette[name] for name in scenario_names]

bars = axes[1].barh(scenario_names, audpc_vals, color=bar_colors, edgecolor="black", alpha=0.85)
axes[1].set_title("Cumulative Area Under Disease Progress Curve (AUDPC)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("AUDPC (Severity-Days)", fontsize=11)
axes[1].grid(True, alpha=0.3)

for bar in bars:
    w = bar.get_width()
    axes[1].text(w + 0.8, bar.get_y() + bar.get_height()/2.0, f"{w:.1f}", ha="left", va="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 09: Numerical Integrity & Statistical Validation
print("[VALIDATE] Validating numerical bounds and stability...")

nan_count = synthetic_df["severity"].isna().sum()
inf_count = np.isinf(synthetic_df["severity"]).sum()
min_val = synthetic_df["severity"].min()
max_val = synthetic_df["severity"].max()

print(f"  - NaN values count        : {nan_count}")
print(f"  - Infinite values count   : {inf_count}")
print(f"  - Minimum severity recorded: {min_val:.5f}")
print(f"  - Maximum severity recorded: {max_val:.5f}")

assert nan_count == 0, "Numerical error: Found NaNs in trajectory output!"
assert inf_count == 0, "Numerical error: Found Infs in trajectory output!"
assert min_val >= 0.0 and max_val <= 1.0, "Biological violation: Severity outside [0, 1]!"
print("  -> All numerical stability assertions passed with 100% compliance.")

# Trajectory-level peak severity analysis
peak_df = synthetic_df.groupby("trajectory_id").agg({
    "severity": ["max", "last"],
    "temperature": "first",
    "humidity": "first",
    "precipitation": "first",
    "infection_rate": "first",
    "latent_period": "first",
    "treated": "first"
})
peak_df.columns = ["peak_severity", "terminal_severity", "temp", "humidity", "precip", "r0", "latent_p", "treated"]

# Correlation Analysis
corr_matrix = peak_df.corr(method="spearman")
print("\nSpearman Correlation with Peak Severity:")
print(corr_matrix["peak_severity"].sort_values(ascending=False))

In [ ]:
# Cell 10: Export Processed Trajectories to CSV
output_csv_path = DATA_PROCESSED_DIR / "vanderplank_synthetic_5k.csv"
print(f"[EXPORT] Saving 5,000 synthetic epidemic trajectories to: {output_csv_path}...")

synthetic_df.to_csv(output_csv_path, index=False)
file_size_mb = output_csv_path.stat().st_size / (1024 * 1024)

print(f"[EXPORT] File written successfully.")
print(f"  - Total rows written: {len(synthetic_df):,}")
print(f"  - Disk footprint    : {file_size_mb:.2f} MB")

In [ ]:
# Cell 11: Summary Statistics & Heatmap Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution of Terminal Severities
axes[0].hist(peak_df["terminal_severity"], bins=30, color="darkolivegreen", edgecolor="black", alpha=0.75)
axes[0].set_title("Distribution of Terminal Epidemic Severity (Day 60)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Severity Index", fontsize=11)
axes[0].set_ylabel("Number of Trajectories", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Heatmap: Temperature vs Relative Humidity vs Mean Terminal Severity
temp_bins = pd.cut(peak_df["temp"], bins=6)
rh_bins = pd.cut(peak_df["humidity"], bins=6)
pivot_table = peak_df.pivot_table(values="terminal_severity", index=rh_bins, columns=temp_bins, aggfunc="mean")

sns.heatmap(pivot_table, ax=axes[1], cmap="YlOrRd", annot=True, fmt=".2f", cbar_kws={'label': 'Mean Terminal Severity'})
axes[1].set_title("Epidemic Severity Risk Matrix: Temp vs RH", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Temperature (°C)", fontsize=11)
axes[1].set_ylabel("Relative Humidity (%)", fontsize=11)

plt.tight_layout()
plt.show()

## Scientific Limitations & Methodological Assumptions

### Synthetic Provenance Disclaimer
The dataset generated in this notebook is synthesized entirely from Van der Plank's non-linear delay-differential epidemiological equations coupled with Gaussian thermal and linear hygrometric response modifiers. While rooted in classical botanical pathology, this data is **not yet validated against empirical multi-year field telemetry** from localized agricultural research extension stations.

### Modeling Assumptions
1. **Uniform Host Canopy Density**: The model assumes a static, homogeneous foliage canopy without dynamic vegetative growth or defoliation during the 60-day epidemic period.
2. **Constant Microclimate**: Daily simulation runs assume stationary temperature, relative humidity, and precipitation values. In real agricultural environments, diurnal microclimatic oscillations and intermittent dew events significantly alter spore germination kinetics.
3. **Exogenous Inoculum Independence**: The simulation treats each field as a closed system without exogenous aerial spore deposition from adjacent farms.

### Downstream Integration in CropSure
- The generated dataset `vanderplank_synthetic_5k.csv` directly provides training sequences for the **BiLSTM Disease Forecaster** in Notebook 03.
- The calculated AUDPC and infection inflection metrics calibrate the **Random Forest Yield Regressor** in Notebook 05.